In [2]:
!git clone https://github.com/ThreadZepplin/llm-training
%cd /kaggle/working/llm-training

Cloning into 'llm-training'...
remote: Enumerating objects: 62, done.
remote: Counting objects: 100% (62/62), done.
remote: Compressing objects: 100% (54/54), done.
remote: Total 62 (delta 7), reused 48 (delta 3), pack-reused 0 (from 0)
Receiving objects: 100% (62/62), 26.88 KiB | 1.49 MiB/s, done.
Resolving deltas: 100% (7/7), done.
/kaggle/working/llm-training


In [3]:
!python -m pip install --upgrade pip
!pip install -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 22.7 MB/s eta 0:00:0000:0100:01
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 630.8/630.8 kB 12.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 121.1 MB/s  0:00:00m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [jupyter]m1/3 [trl]


In [4]:
from pathlib import Path
import shutil

repo_root = Path("/kaggle/working/llm-training")
raw_dir = repo_root / "data" / "raw"
raw_dir.mkdir(parents=True, exist_ok=True)

copied = []

for dataset_dir in Path("/kaggle/input").iterdir():
    if dataset_dir.is_dir():
        print("Checking dataset folder:", dataset_dir)
        for path in dataset_dir.rglob("*"):
            if path.is_file() and path.suffix.lower() == ".csv":
                shutil.copy(path, raw_dir / path.name)
                copied.append(path.name)

print("Copied files to:", raw_dir)
print("Copied", len(copied), "CSV files")
print(copied)

Checking dataset folder: /kaggle/input/datasets
Copied files to: /kaggle/working/llm-training/data/raw
Copied 25 CSV files
['7.5in x 8in Hex1 2xBrim Front Right-20250925083521.csv', '7.5in x 8in Hex1 2xBrim Front Right-20250926080727.csv', '7.5in x 8in Hex1 2xBrim Front Right-20251007080848.csv', '7.5in x 8in Hex2 2xBrim Front Left-20250916111958.csv', '017_Sample_033_ros.csv', '7.5in x 8in Hex 2xBrim Rear Right PolyPro-20250924083722.csv', '7.5in x 8in Hex1 2xBrim Front Right-20250915080848.csv', '7.5in x 8in Hex1 2xBrim Front Right-20251002084414.csv', '7.5in x 8in Hex 2xBrim Rear Left PolyPro-20251003080433.csv', '7.5in x 8in Hex1 2xBrim Front Right-20250916080815.csv', '7.5in x 8in Hex 2xBrim Rear Right PolyPro-20251001130529.csv', '7.5in x 8in Hex1 2xBrim Front Right-20250930081421.csv', '7.5in x 8in Hex1 2xBrim Front Right-20250929081954.csv', '7.5in x 8in Hex 2xBrim Rear Left PolyPro-20250924115019.csv', '015_Sample_031_ros.csv', '7.5in x 8in Hex2 2xBrim Front Left-2025091512413

In [5]:
!python scripts/preprocess_data.py --input-dir data/raw --output-file data/processed/operations.jsonl

Wrote 25 records to data/processed/operations.jsonl


In [6]:
!python scripts/generate_synthetic_records.py --input-file data/processed/operations.jsonl --output-file data/processed/synthetic_operator_records.jsonl

Wrote 21 synthetic operator records to data/processed/synthetic_operator_records.jsonl


In [7]:
!python scripts/build_dataset_splits.py --real-file data/processed/operations.jsonl --synthetic-file data/processed/synthetic_operator_records.jsonl --out-all data/processed/dataset_all.jsonl --out-train data/processed/dataset_train.jsonl --out-test data/processed/dataset_test_manual.jsonl --out-meta data/processed/dataset_split_metadata.json

Wrote 46 total examples to data/processed/dataset_all.jsonl
Wrote 36 train examples to data/processed/dataset_train.jsonl
Wrote 10 test examples to data/processed/dataset_test_manual.jsonl
Wrote split metadata to data/processed/dataset_split_metadata.json
Held-out groups:
  - 015_Sample_023_ros.csv
  - 015_Sample_031_ros.csv
  - 7.5in x 8in Hex 2xBrim Rear Left PolyPro-20250924115019.csv
  - 7.5in x 8in Hex 2xBrim Rear Left PolyPro-20251003080433.csv
  - 7.5in x 8in Hex 2xBrim Rear Left-20251010095359.csv
  - 7.5in x 8in Hex 2xBrim Rear Right PolyPro-20251001130529.csv


In [8]:
!python scripts/prepare_finetune_files.py --train-input data/processed/dataset_train.jsonl --test-input data/processed/dataset_test_manual.jsonl --out-train data/processed/finetune_train.jsonl --out-test-inputs data/processed/finetune_test_inputs.jsonl --out-test-gold data/processed/finetune_test_gold.jsonl

Wrote 36 training rows to data/processed/finetune_train.jsonl
Wrote 10 test input rows to data/processed/finetune_test_inputs.jsonl
Wrote 10 test gold rows to data/processed/finetune_test_gold.jsonl


In [9]:
!python scripts/train_mistral_lora.py --config configs/mistral_lora.json

config.json: 100%|█████████████████████████████| 601/601 [00:00<00:00, 2.91MB/s]
tokenizer_config.json: 141kB [00:00, 39.2MB/s]
tokenizer.json: 1.96MB [00:00, 43.3MB/s]
tokenizer.model: 100%|███████████████████████| 587k/587k [00:00<00:00, 1.27MB/s]
special_tokens_map.json: 100%|█████████████████| 414/414 [00:00<00:00, 2.40MB/s]
Generating train split: 36 examples [00:00, 1747.63 examples/s]
Map: 100%|██████████████████████████████| 36/36 [00:00<00:00, 849.80 examples/s]
`torch_dtype` is deprecated! Use `dtype` instead!
model.safetensors.index.json: 23.9kB [00:00, 68.7MB/s]
Fetching 3 files: 100%|███████████████████████████| 3/3 [00:56<00:00, 18.69s/it]
Download complete: 100%|████████████████████| 14.5G/14.5G [00:56<00:00, 210MB/s]
Loading weights:   0%|                                  | 0/291 [00:00<?, ?it/s]
Loading weights:   0%| | 1/291 [00:00<00:00, 10280.16it/s, Materializing param=l
Loading weights:   0%| | 1/291 [00:00<00:00, 2668.13it/s, Materializing param=lm
Loading weight

In [10]:
!python scripts/eval_mistral_lora.py --base-model mistralai/Mistral-7B-Instruct-v0.3 --adapter-path outputs/mistral7b_lora_manufacturing --test-inputs data/processed/finetune_test_inputs.jsonl --output-file outputs/mistral7b_test_predictions.jsonl

`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|█| 291/291 [00:07<00:00, 40.78it/s, Materializing param=mo
ex_0005: 31.48s
ex_0009: 27.79s
ex_0011: 33.24s
ex_0014: 28.34s
ex_0022: 31.67s
ex_0025: 32.96s
ex_0030: 36.13s
ex_0035: 35.62s
ex_0044: 35.88s
ex_0046: 35.18s

Wrote predictions to outputs/mistral7b_test_predictions.jsonl
Total inference time: 328.29s
Average time per example: 32.83s


In [11]:
!python scripts/export_prediction_summaries.py outputs/mistral7b_test_predictions.jsonl

Wrote summaries to outputs/mistral7b_test_predictions.txt
